In [8]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from dotenv import load_dotenv
from typing import TypedDict,Literal,Dict,List,Annotated,Optional
from pydantic import BaseModel,Field
load_dotenv()
model=ChatOpenAI()


python-dotenv could not parse statement starting at line 5


In [9]:
class Sentiment_State(StateGraph):
    review:str
    Diagnosis:Dict
    sentiment:Literal['positive','negative']
    response:str


In [10]:
class SentimentSchema(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="generate the sentiment fot the input")

Structured_model=model.with_structured_output(SentimentSchema)


In [12]:
def Find_sentiment(state:Sentiment_State):
    review=state['review']
    result=Structured_model.invoke(review)
    return {'sentiment':result.sentiment}

graph=StateGraph(Sentiment_State)
graph.add_node('Find_sentiment',Find_sentiment)
graph.add_edge(START,'Find_sentiment')
graph.add_edge('Find_sentiment',END)
workflow=graph.compile()
initial_state={'review':"I don't like this mobile"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'review': "I don't like this mobile", 'sentiment': 'negative'}
